# CUDA Kernel Ladder — from vector add to fused attention

Write **real CUDA C++** on a **free NVIDIA T4** GPU, straight from your Mac's browser.

**First, get the GPU:** `Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save`.
Then run the cells top to bottom.

The ladder: `vector add → tiled matmul → reduction → softmax → fused attention`.
That exact sequence is the GPU-systems interview / RA skillset. Kernels 1–2 are fully written;
3–5 are stubs with the problem stated — fill them in as you climb.

In [ ]:
# 1. Confirm the GPU is attached. You should see a Tesla T4 (~15 GB).
!nvidia-smi

In [ ]:
# 2. nvcc4jupyter lets a cell of CUDA C++ compile with nvcc and run on the GPU.
!pip install -q nvcc4jupyter
%load_ext nvcc4jupyter

## Kernel 1 — Vector add  ·  *thread indexing, host↔device memory*

The "hello world" of CUDA. One thread per element. The whole mental model is here:
- `blockIdx.x * blockDim.x + threadIdx.x` maps a thread to a global index
- you `cudaMalloc` on the device, `cudaMemcpy` across the PCIe bus, launch `<<<blocks, threads>>>`, copy back
- the `if (i < n)` guard handles the ragged last block

In [ ]:
%%cuda
#include <stdio.h>

__global__ void vecAdd(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main() {
    const int N = 1 << 20;                 // ~1M elements
    size_t bytes = N * sizeof(float);

    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);
    for (int i = 0; i < N; i++) { h_a[i] = 1.0f; h_b[i] = 2.0f; }

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes); cudaMalloc(&d_b, bytes); cudaMalloc(&d_c, bytes);
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks  = (N + threads - 1) / threads;
    vecAdd<<<blocks, threads>>>(d_a, d_b, d_c, N);
    cudaDeviceSynchronize();

    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);
    printf("c[0] = %.1f, c[N-1] = %.1f  (expect 3.0)\n", h_c[0], h_c[N-1]);

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);
    return 0;
}

## Kernel 2 — Tiled matmul  ·  *shared memory, `__syncthreads`, data reuse*

This is the one to internalise. A naive matmul re-reads global memory O(N) times per output;
**tiling** stages `TILE×TILE` blocks of A and B into on-chip **shared memory**, so each loaded
value is reused by a whole tile of threads. That reuse *is* the idea behind systolic arrays —
reimplement your NPU-project matmul here and you've connected the accelerator work to real CUDA.

Try: flip `TILE` between 8/16/32 and watch the timing move.

In [ ]:
%%cuda
#include <stdio.h>

#define TILE 16

__global__ void matmulTiled(const float* A, const float* B, float* C, int N) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;

    float acc = 0.0f;
    for (int t = 0; t < N / TILE; t++) {
        As[threadIdx.y][threadIdx.x] = A[row * N + (t * TILE + threadIdx.x)];
        Bs[threadIdx.y][threadIdx.x] = B[(t * TILE + threadIdx.y) * N + col];
        __syncthreads();                      // wait for the tile to be fully loaded

        for (int k = 0; k < TILE; k++)
            acc += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        __syncthreads();                      // wait before overwriting the tile
    }
    C[row * N + col] = acc;
}

int main() {
    const int N = 512;                        // multiple of TILE
    size_t bytes = N * N * sizeof(float);

    float *h_A = (float*)malloc(bytes);
    float *h_B = (float*)malloc(bytes);
    float *h_C = (float*)malloc(bytes);
    for (int i = 0; i < N * N; i++) { h_A[i] = 1.0f; h_B[i] = 1.0f; }

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes); cudaMalloc(&d_B, bytes); cudaMalloc(&d_C, bytes);
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);

    dim3 threads(TILE, TILE);
    dim3 blocks(N / TILE, N / TILE);

    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);
    matmulTiled<<<blocks, threads>>>(d_A, d_B, d_C, N);
    cudaEventRecord(stop); cudaEventSynchronize(stop);
    float ms = 0; cudaEventElapsedTime(&ms, start, stop);

    cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);
    printf("C[0] = %.1f (expect %d)   |   %.3f ms\n", h_C[0], N, ms);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);
    return 0;
}

## Kernel 3 — Parallel reduction (sum)  ·  *tree reduction, warp divergence*  🔨 *your turn*

**Goal:** sum an array of N floats. A single thread doing `for` is O(N) serial — instead reduce a
block in shared memory in `log2(blockDim)` steps, then combine block partials.

**Milestones (each is a real optimisation lesson):**
1. Naive tree reduction in shared memory with `s *= 2` strides — works, but half the warp idles.
2. Reverse the stride (`s /= 2`) so active threads stay contiguous — kills warp divergence.
3. Unroll the last warp / use `__shfl_down_sync` warp shuffles — no shared memory for the tail.

Start from the vector-add skeleton; keep a CPU `std::accumulate` as your correctness check.

In [ ]:
%%cuda
#include <stdio.h>

// TODO: __global__ void reduceSum(const float* in, float* blockSums, int n) { ... }
// Hint: extern __shared__ float sdata[]; load, __syncthreads(), then the reduction loop.

int main() {
    printf("Kernel 3 — implement reduceSum, then sum the block partials on the host.\n");
    return 0;
}

## Kernel 4 — Softmax (row-wise)  ·  *numerical stability, two-pass reduction*  🔨 *your turn*

**Goal:** softmax over each row of an MxN matrix. Per row: subtract the row max (stability), exp,
divide by the row sum. That's a **max-reduction** and a **sum-reduction** per row — you're reusing
Kernel 3's machinery. This is the direct warm-up for attention.

## Kernel 5 — Fused attention  ·  *the payoff*  🔨 *your turn*

**Goal:** one kernel computes `softmax(Q·Kᵀ / √d) · V` without writing the full scores matrix back
to global memory. Combine Kernel 2 (matmul) + Kernel 4 (softmax) and keep intermediates on-chip —
the core idea behind FlashAttention. This is the memorable line on a systems-ML resume.

## Where to go next

- **Benchmark Kernel 2 against cuBLAS** (`cublasSgemm`) — expect cuBLAS to win big; understanding
  *why* (tensor cores, better tiling/scheduling) is the real lesson. Log numbers in `benchmarks/`.
- **Extract kernels to `.cu` files** in `kernels/` and compile with `nvcc file.cu -o out` once you
  outgrow notebook cells.
- **Profile** with `nvidia-smi` and, later, Nsight Compute to see occupancy and memory throughput.

See the repo `README.md` for the week-by-week plan and resources (PMPP, GPU MODE, Triton, llm.c).